# 11 — BERTopic del corpus español con stopwords (corrección)

El BERTopic ejecutado dentro del notebook 07 produjo tópicos cuyas palabras
clave eran mayoritariamente palabras vacías en español (de, la, el, que, por...),
lo que los hacía inservibles para la interpretación.

Este notebook rehace **únicamente** la fase BERTopic del corpus español,
añadiendo un `CountVectorizer` con stopwords en español para que las palabras
clave de cada tópico sean términos con significado.

## Importante
- NO se toca la clasificación zero-shot (las columnas `predicted_category`,
  `confidence_score`, `is_relevant` se conservan intactas).
- Solo se regeneran `bertopic_id` y `bertopic_keywords`.
- Entrada y salida: `scam_es_FINAL_classified_corregido.csv` (se actualiza in situ
  tras guardar copia de seguridad).

⏱️ ~5-10 minutos.


In [ ]:
import pandas as pd
import re
import time

pd.set_option('display.max_colwidth', None)

df = pd.read_csv("../data/raw/scam_es_FINAL_classified_corregido.csv")
print(f"Tweets cargados: {len(df)}")

# Texto limpio para BERTopic
URL_RE = re.compile(r"https?://\S+")
MENTION_RE = re.compile(r"@\w+")
def clean(t):
    if not isinstance(t,str): return ""
    t = URL_RE.sub("", t)
    t = MENTION_RE.sub("", t)
    return re.sub(r"\s+"," ",t).strip()

if "clean_text" not in df.columns or df["clean_text"].isna().all():
    df["clean_text"] = df["text"].apply(clean)
else:
    df["clean_text"] = df["clean_text"].fillna("").apply(lambda x: x if x else "")
    m = df["clean_text"]==""
    df.loc[m,"clean_text"] = df.loc[m,"text"].apply(clean)

docs = df["clean_text"].tolist()
print(f"Documentos para BERTopic: {len(docs)}")


## Lista de stopwords en español

Se combina la lista de NLTK (si está disponible) con una lista manual de
refuerzo de términos vacíos frecuentes en tweets.


In [ ]:
# Stopwords base en español
try:
    from nltk.corpus import stopwords
    import nltk
    try:
        ES_STOP = stopwords.words("spanish")
    except LookupError:
        nltk.download("stopwords")
        ES_STOP = stopwords.words("spanish")
    print(f"Stopwords NLTK español: {len(ES_STOP)}")
except Exception as e:
    print(f"NLTK no disponible ({e}); se usa solo lista manual")
    ES_STOP = []

# Refuerzo manual: vacías y ruido típico de tweets
EXTRA = [
    "de","la","el","que","en","y","a","los","las","un","una","unos","unas",
    "por","con","para","su","sus","se","del","al","lo","le","les","es","son",
    "este","esta","esto","estos","estas","ese","esa","eso","como","más","mas",
    "ya","si","no","te","me","mi","tu","nos","muy","ha","han","hay","sobre",
    "https","http","co","rt","via","t","q","x","d","pa","cuando","donde",
]
ES_STOP_FULL = sorted(set([w.lower() for w in ES_STOP] + EXTRA))
print(f"Stopwords totales (NLTK + refuerzo): {len(ES_STOP_FULL)}")


## BERTopic con CountVectorizer que filtra stopwords

In [ ]:
print("⏳ Ejecutando BERTopic con stopwords en español...")
t0 = time.time()

from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# El CountVectorizer se aplica en la fase de extracción de keywords:
# elimina stopwords y exige que un término aparezca en al menos 2 documentos.
vectorizer = CountVectorizer(
    stop_words=ES_STOP_FULL,
    min_df=2,
    ngram_range=(1, 2),
)

topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2",
    vectorizer_model=vectorizer,
    min_topic_size=20,
    nr_topics="auto",
    calculate_probabilities=False,
    language="multilingual",
    verbose=True,
)

topics, _ = topic_model.fit_transform(docs)

t = time.time() - t0
n_top = len(set(topics)) - (1 if -1 in topics else 0)
n_out = sum(1 for x in topics if x == -1)
print(f"\n✓ BERTopic completado en {t/60:.1f} min")
print(f"  Tópicos: {n_top} | Outliers: {n_out}")


In [ ]:
# Actualizar columnas en el dataframe
df["bertopic_id"] = topics

topic_keywords = {}
for tid in set(topics):
    if tid == -1:
        topic_keywords[tid] = "outliers"
        continue
    words = topic_model.get_topic(tid)
    topic_keywords[tid] = ", ".join([w for w,_ in words[:8]]) if words else ""

df["bertopic_keywords"] = df["bertopic_id"].map(topic_keywords)

print("=== TÓPICOS BERTOPIC ESPAÑOL (con stopwords filtradas) ===\n")
for tid in sorted(set(topics)):
    n = (df["bertopic_id"]==tid).sum()
    print(f"  Tópico {tid:>3} ({n:>4} tweets): {topic_keywords.get(tid,'')[:75]}")


In [ ]:
print("=== EJEMPLOS POR TÓPICO ===\n")
shown = 0
for tid in sorted(set(topics)):
    if tid == -1 or shown >= 10:
        continue
    sub = df[df["bertopic_id"]==tid]
    print(f"--- TÓPICO {tid} ({len(sub)} tweets) — {topic_keywords.get(tid,'')[:60]} ---")
    for _, r in sub.head(2).iterrows():
        print(f"   • {str(r['text'])[:170]}")
    print()
    shown += 1


## Guardado

Copia de seguridad del CSV anterior y guardado del actualizado.


In [ ]:
import shutil
# Backup por seguridad
shutil.copy("../data/raw/scam_es_FINAL_classified_corregido.csv",
            "../data/raw/scam_es_FINAL_classified_corregido_pre_bertopic.csv")
print("✓ Backup: scam_es_FINAL_classified_corregido_pre_bertopic.csv")

# Guardar el actualizado
df.to_csv("../data/raw/scam_es_FINAL_classified_corregido.csv", index=False, encoding="utf-8")
print("✓ Actualizado: scam_es_FINAL_classified_corregido.csv")
print()
print("Columnas bertopic_id y bertopic_keywords regeneradas.")
print("La clasificación zero-shot (predicted_category) NO se ha modificado.")
print()
print("🚨 Sube la versión actualizada a Drive.")
